# 01 · The RAG skeleton: packages, imports and code

The interviewer hands you a folder of documents and a blank editor: *"build a simple RAG app"*. By the end of this notebook you can write it from memory. You will know every `pip install`, every import line and what each one is for, and you will have an answer ready for *"what would you add second?"*

**Time:** ~35 min · **Runs:** offline on CPU in about a minute (the first import cell is the slow part) · **Needs:** nothing. [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb) goes deeper on each piece.

## Why this matters in interviews

- "Here is a folder, build RAG over it" is the most common hands-on GenAI round. It is usually 30–45 minutes with a shared editor.
- They grade the **structure** (index time vs query time), your **package choices and swaps**, a **grounded prompt with citations**, and whether you handle **empty or irrelevant retrieval**.
- LangChain 1.x moved classes around. `from langchain.retrievers import EnsembleRetriever` now raises `ModuleNotFoundError`, and knowing the new path saves you an awkward minute on a live screen share.
- The follow-up *"what would you add second?"* tells apart the people who have shipped RAG from the people who have only watched a tutorial.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg00` | Design a RAG system over a knowledge base folder: what language, what packages, and what does the skeleton code look like? |
| `tl15` | Name the pieces of a RAG pipeline and which one usually breaks. |
| `rg07` | What is hybrid search and how do you combine the two lanes? |
| `rg45` | What is reciprocal rank fusion and why is it the default? |
| `rg25` | How do you handle empty or low-confidence retrieval? |
| `rg08` | What is reranking, and why can a reranker not fix bad recall? |
| `rg12` | How do you build a RAG eval set from nothing? |
| `rg29` | How do you implement permission-aware retrieval? |
| `rg40` | How do you keep the index and the source of truth in sync? |
| `pd10` | Why does LangChain (or any framework) exist, if the providers already have APIs? |

## The question, as the interviewer asks it

> *"Here's `knowledge_base/`, a folder of markdown files from our support team. Build me a simple RAG app over it: a function that takes a question and returns an answer with its sources. Any language, any packages. Talk me through it as you go. You have 30 minutes."*

## The first 60 seconds: what to say before you type

**Ask two or three clarifying questions, not ten:**

| Ask | Why it changes the code |
|---|---|
| How many files, and what formats? | 50 markdown files fit in an in-process FAISS index. Five million PDFs need a managed vector DB and an ingestion pipeline. |
| Must answers cite their sources? | Assume yes. It decides what metadata you keep on every chunk. |
| Hosted API allowed, or must data stay on our machines? | OpenAI embeddings and chat, or a local sentence-transformers model plus Ollama. |
| Can every user see every document? | If not, permission filters go *into retrieval* (follow-up 3 below). |
| How often do documents change? | Decides whether rebuilding the index at startup is fine (follow-up 4). |

**Then say the plan out loud (about 20 seconds):**

> "It splits into two halves. **Index time**, once: load the folder, split it into roughly 800-character chunks with some overlap, embed each chunk, and save the vectors in a FAISS index. **Query time**, on every question: retrieve through two lanes, vectors for meaning and BM25 for exact words like error codes, and fuse them with reciprocal rank fusion. Then put the top chunks into a prompt that says *answer only from this, cite the filename, say you don't know otherwise*, and return the answer with its sources. I'll write it in Python with LangChain as the glue so each piece can be swapped, FAISS in-process, and OpenAI for both models. After that I'll say what I'd add second."

That paragraph already answers `tl15`. The next cell draws it.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model, langchain_embeddings, OFFLINE

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

draw_pipeline(["load folder\nDirectoryLoader", "split\n800 / 120 chars", "embed\nOpenAIEmbeddings",
               "store\nFAISS (on disk)"], title="INDEX TIME: once, and again when a document changes")
draw_pipeline(["question", "retrieve\ndense + BM25", "fuse\nRRF", "prompt\ncontext + rules",
               "generate\nChatOpenAI", "answer\n+ sources"], title="QUERY TIME: on every question")

## Language and packages

**Language: Python 3.11+.** Every vector store, model SDK and retrieval library ships its Python client first, so the whole stack is one `pip install`:

```
pip install langchain langchain-community langchain-text-splitters langchain-openai langchain-classic faiss-cpu rank_bm25
# env: OPENAI_API_KEY
```

| Package (`pip` name) | What you import from it | What it gives you | Swap it for |
|---|---|---|---|
| `langchain` | nothing directly here | The umbrella package. It installs `langchain-core`, which holds the shared interfaces (`Document`, `ChatPromptTemplate`, `StrOutputParser`, the `\|` pipe, the retriever and embeddings base classes). It also gives you `create_agent` for later. | LlamaIndex or Haystack, or no framework at all (see the 40-line version below) |
| `langchain-community` | `DirectoryLoader`, `TextLoader`, `FAISS`, `BM25Retriever` | Document loaders for many file types, the FAISS vector-store wrapper and the BM25 retriever | `unstructured` or `pypdf` for PDFs. The standalone integration packages (`langchain-chroma`, ...) are replacing it: it now prints a *"being sunset"* `DeprecationWarning` but still works |
| `langchain-text-splitters` | `RecursiveCharacterTextSplitter` | Splits on paragraph, then line, then word boundaries, with overlap | `MarkdownHeaderTextSplitter` once headings matter, or a token-based splitter |
| `langchain-openai` | `OpenAIEmbeddings`, `ChatOpenAI` | The embedding model and the chat model as LangChain objects | Local embeddings: `HuggingFaceEmbeddings` (`langchain-huggingface`, runs sentence-transformers). Local chat: `ChatOllama` (`langchain-ollama`). Claude: `ChatAnthropic` (`langchain-anthropic`) |
| `langchain-classic` | `EnsembleRetriever` | Weighted reciprocal rank fusion of several retrievers. It **moved here in LangChain 1.x** | Ten lines of your own RRF (below) |
| `faiss-cpu` | used by `FAISS` | The vector index: in-process, exact or approximate, saved to a folder | Chroma (`langchain-chroma`), Qdrant (`langchain-qdrant`), pgvector (`langchain-postgres`) |
| `rank_bm25` | used by `BM25Retriever` | BM25 keyword scoring | Elasticsearch or OpenSearch for large corpora, or SPLADE sparse vectors |

The swaps really are one line each, because every store and every model implements the same interface. These are the real APIs, shown as text here because the packages are not installed in this venv:

```python
from langchain_chroma import Chroma                     # pip install langchain-chroma
store = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")

from langchain_qdrant import QdrantVectorStore          # pip install langchain-qdrant ; a Qdrant server
store = QdrantVectorStore.from_documents(chunks, embeddings, url="http://localhost:6333", collection_name="kb")

from langchain_huggingface import HuggingFaceEmbeddings # pip install langchain-huggingface ; local, free
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

from langchain_ollama import ChatOllama                 # pip install langchain-ollama ; `ollama pull qwen2.5:3b`
llm = ChatOllama(model="qwen2.5:3b", temperature=0)
```

## The imports, with a comment on every line

This is the top of `rag.py`. The cell is slow the first time (10–30 s). `langchain-text-splitters` eagerly imports `sentence-transformers` and `transformers` whenever they are installed, and they are in this venv. That cost has nothing to do with your code.

In [ ]:
# --- standard library
import re                           # the BM25 tokenizer (a regex)
import tempfile                     # a scratch folder, so this notebook never writes into the repo

# --- index time: read files, cut them into chunks
from langchain_community.document_loaders import DirectoryLoader, TextLoader  # folder -> list[Document]
from langchain_text_splitters import RecursiveCharacterTextSplitter           # Document -> overlapping chunks

# --- the two retrieval lanes and the fusion
from langchain_community.vectorstores import FAISS          # embed + index + nearest-neighbour search, save/load
from langchain_community.retrievers import BM25Retriever    # keyword lane (rank_bm25 under the hood)
from langchain_classic.retrievers import EnsembleRetriever  # weighted reciprocal rank fusion of the lanes

# --- query time: prompt -> model -> string
from langchain_core.prompts import ChatPromptTemplate       # the prompt, with {question} and {context} slots
from langchain_core.output_parsers import StrOutputParser   # AIMessage -> plain str at the end of the chain
from langchain_core.documents import Document               # (page_content, metadata): what flows between steps
from langchain_core.embeddings import Embeddings            # the 2-method interface every embedder implements

# --- the models: in rag.py you construct these two directly
from langchain_openai import ChatOpenAI, OpenAIEmbeddings   # OpenAI chat + embedding models as LangChain objects

print("all imports OK")

**The gotcha to say out loud:** the card-era import no longer exists. In LangChain 1.x the legacy chains and retrievers moved to `langchain-classic`, and `langchain` itself became a slim package built around agents.

In [ ]:
try:
    from langchain.retrievers import EnsembleRetriever as OldEnsemble   # the pre-1.0 path
    old_path_works = True
except ModuleNotFoundError as err:
    old_path_works = False
    print("old import ->", type(err).__name__ + ":", err)

assert not old_path_works, "LangChain 1.x: use `from langchain_classic.retrievers import EnsembleRetriever`"
print("new import  ->", EnsembleRetriever.__module__)

## Step 0 · A tiny knowledge base

Eight short support articles in a temp folder. One of them, `internal/refund_exceptions.md`, is for staff only. The skeleton will happily index it anyway, and follow-up 3 comes back to that. `faq.md` is long enough to be split into several chunks.

In [ ]:
WORK = Path(tempfile.mkdtemp(prefix="rag_skeleton_"))
KB_FILES = {
    "refunds.md": """# Refund policy

You can ask for a refund within 30 days of delivery.
Refunds go back to the original payment method within 5 business days of approval.
Items marked final sale cannot be returned or refunded.
To start a refund, open Orders, pick the item and click "Request refund".""",
    "shipping.md": """# Shipping

Standard shipping takes 3 to 5 business days and is free on orders over $50.
Express shipping arrives the next business day and costs $15.
We ship to the United States, Canada and the European Union.
Orders placed after 2 pm leave the warehouse on the next business day.""",
    "error_codes.md": """# Checkout error codes

Error E-4012 means the bank declined the card; ask the customer to use another card.
Error E-5003 means the warehouse API timed out; wait two minutes and retry the order.
Error E-7001 means the shipping address failed validation; check the postcode and country.""",
    "account_security.md": """# Account security

Password reset links expire after 30 minutes.
After 5 failed login attempts the account is locked for 15 minutes.
Two-factor authentication is switched on under Settings, then Security.
Support staff will never ask a customer for their password.""",
    "warranty.md": """# Warranty

All electronics come with a 2-year warranty that covers manufacturing defects.
The warranty does not cover accidental damage, water damage or normal wear.
To make a warranty claim, send the serial number and a photo through the support portal.""",
    "pricing_plans.md": """# Pricing plans

The Basic plan costs $9 per month and includes 3 projects.
The Pro plan costs $29 per month and includes unlimited projects and priority support.
Paying yearly saves 20% on either plan.
You can cancel at any time; the plan stays active until the end of the billing period.""",
    "faq.md": """# Frequently asked questions

Q: Can I change my delivery address after ordering?
A: Yes, until the order ships. Open Orders and choose "Edit address".
Q: Do you offer gift wrapping?
A: Yes, gift wrapping costs $4 per item and is chosen at checkout.
Q: Can I order by phone?
A: No, orders are placed online only, but phone support can walk you through checkout.
Q: How do I apply a discount code?
A: Type the code into the "Promo code" box on the payment page before you pay.
Q: Can I combine two discount codes?
A: No, only one discount code can be used per order.
Q: Do you price match?
A: We match the price of an identical item sold by a major retailer within 7 days of purchase.
Q: How do I delete my account?
A: Go to Settings, then Privacy, and choose "Delete account"; deletion finishes within 30 days.
Q: Where can I download an invoice?
A: Every order page has a "Download invoice" button once the order ships.
Q: Do you sell gift cards?
A: Yes, digital gift cards from $10 to $500, valid for 3 years.
Q: Is my card number stored?
A: Card numbers are kept by our payment provider, never on our own servers.
Q: Can I pause a subscription?
A: Yes, a subscription can be paused for up to 3 months from the Billing page.""",
    "internal/refund_exceptions.md": """# Refund exceptions (internal only)

Agents may approve a refund of up to $200 outside the 30-day window.
Refund exceptions above $200 need sign-off from a team lead.
Do not share this exception policy with customers.""",
}
for rel, text in KB_FILES.items():
    path = WORK / "knowledge_base" / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text + "\n", encoding="utf-8")

for p in sorted((WORK / "knowledge_base").rglob("*.md")):
    print(f"{str(p.relative_to(WORK / 'knowledge_base')):32} {p.stat().st_size:5d} bytes")

## Step 1 · `build_chunks()`: load and split

From here on, the functions are the ones in `rag.py`, line for line. The only difference is where `llm` and `embeddings` come from (Step 2).

- `DirectoryLoader(..., glob="**/*.md", loader_cls=TextLoader)` reads every markdown file, including those in subfolders. Each becomes a `Document` with `metadata["source"]` set to its full path.
- `loader_kwargs={"encoding": "utf-8"}` matters on Windows. Without it, `TextLoader` decodes with the locale codepage (cp1252) and an em dash quietly turns into `â€”`.
- `chunk_size=800, chunk_overlap=120` are **characters**, not tokens (about 200 and 30 tokens). They are a sensible day-one guess and nothing more. The eval set should get the final say.
- `add_start_index=True` stores each chunk's character offset in its file, so a UI can highlight the passage.

In [ ]:
KB_DIR = WORK / "knowledge_base"
DB_DIR = WORK / "faiss_index"
TOP_K = 4

def build_chunks(kb_dir=KB_DIR):
    docs = DirectoryLoader(str(kb_dir), glob="**/*.md", loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=120, add_start_index=True
    ).split_documents(docs)
    for c in chunks:                              # the filename is what the answer will cite
        c.metadata["source"] = Path(c.metadata["source"]).name
    return chunks

chunks = build_chunks()
print(f"{len(KB_FILES)} files -> {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"#{i:<2} {c.metadata['source']:24} start={c.metadata['start_index']:4}  chars={len(c.page_content):4}")
assert all(len(c.page_content) <= 800 for c in chunks)

In [ ]:
faq = [c for c in chunks if c.metadata["source"] == "faq.md"]
a, b = faq[-2], faq[-1]
overlap = a.metadata["start_index"] + len(a.page_content) - b.metadata["start_index"]
print(f"faq.md chunks overlap by {overlap} characters. The line both chunks share:")
print("   ", repr(b.page_content.splitlines()[0]))
assert overlap > 0 and b.page_content.splitlines()[0] in a.page_content

labels = [f"{c.metadata['source']} #{i}" for i, c in enumerate(chunks)]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(labels, [len(c.page_content) for c in chunks], color="#4c6ef5")
ax.axvline(800, color="crimson", ls="--", label="chunk_size = 800")
ax.invert_yaxis(); ax.set_xlabel("characters"); ax.legend(loc="lower right")
ax.set_title("Chunk lengths: one chunk per short file, faq.md split in three")
plt.tight_layout(); plt.show()

Read the chart and the table together. There are two things to say in the interview:

1. **Overlap works on line boundaries.** The splitter cut `faq.md` between lines, and the last line or two of one chunk are repeated at the top of the next. A fact that sits on the cut is therefore whole in at least one chunk.
2. **The 28-character chunk is only the heading** (`# Frequently asked questions`). Heading-only chunks carry no facts and waste a retrieval slot. The fixes are `MarkdownHeaderTextSplitter`, merging tiny chunks into the next one, or prepending the heading path to every chunk (`rg41`).

## Step 2 · `build_store()`: embed, index, save, load

`FAISS.from_documents(chunks, embeddings)` embeds every chunk in batched API calls and builds an exact index in memory. `save_local` writes two files:

| File | Contents |
|---|---|
| `index.faiss` | the vectors, in FAISS's binary format |
| `index.pkl` | the chunk texts, their metadata and the id map, **pickled** |

Unpickling can run arbitrary code, so `FAISS.load_local` refuses to load unless you pass `allow_dangerous_deserialization=True`. Pass it only for files **your own process wrote**, never for an index someone uploaded. And always load with the **same embedding model** you built with. Vectors from two different models live in different spaces (`rg14`).

In `rag.py` the two models are one line each: `OpenAIEmbeddings(model="text-embedding-3-small")` and `ChatOpenAI(model="gpt-4.1-mini", temperature=0)`. Here they come from `llm_setup`, which builds exactly those objects when `OPENAI_API_KEY` is set, and otherwise points the same classes at a local Ollama server or the offline stand-in. The embedder is also wrapped in a 10-line counter, because embedding calls are what you pay for at index time.

In [ ]:
class CountingEmbeddings(Embeddings):
    """Wraps any LangChain embedder and counts the texts it embeds."""
    def __init__(self, inner):
        self.inner, self.texts_embedded = inner, 0
    def embed_documents(self, texts):
        self.texts_embedded += len(texts)
        return self.inner.embed_documents(texts)
    def embed_query(self, text):
        self.texts_embedded += 1
        return self.inner.embed_query(text)

embeddings = CountingEmbeddings(langchain_embeddings())   # rag.py: OpenAIEmbeddings(model="text-embedding-3-small")
llm = langchain_chat_model(temperature=0)                  # rag.py: ChatOpenAI(model="gpt-4.1-mini", temperature=0)
assert isinstance(llm, ChatOpenAI) and isinstance(embeddings.inner, OpenAIEmbeddings)

def build_store(chunks, db_dir=DB_DIR):
    if db_dir.exists():                           # already embedded: do not pay for it twice
        return FAISS.load_local(str(db_dir), embeddings,
                                allow_dangerous_deserialization=True)  # our own file, so safe
    store = FAISS.from_documents(chunks, embeddings)
    store.save_local(str(db_dir))
    return store

store = build_store(chunks)
print(f"built: {store.index.ntotal} vectors of dim {store.index.d}; texts embedded so far: {embeddings.texts_embedded}")
print("files on disk:", sorted(p.name for p in DB_DIR.iterdir()))

In [ ]:
before = embeddings.texts_embedded
store = build_store(chunks)                       # second call: loads from disk
assert embeddings.texts_embedded == before, "loading must not re-embed anything"
print(f"loaded from disk: {store.index.ntotal} vectors, 0 new embedding calls")

try:
    FAISS.load_local(str(DB_DIR), embeddings)     # forgot the flag
except ValueError as err:
    print("\nwithout the flag ->", str(err)[:120], "...")

hits = store.similarity_search_with_score("How long do password reset links last?", k=3)
for doc, dist in hits:
    print(f"  L2 distance {dist:.3f}  {doc.metadata['source']}")
assert hits[0][1] <= hits[-1][1]

A common trap is hiding in those numbers. **FAISS returns distances, and lower means closer.** The LangChain wrapper uses L2 distance by default. OpenAI embeddings are unit-length, so ranking by L2 gives the same order as ranking by cosine similarity, but the scores run the other way. If you want a 0–1 "higher is better" value, use `similarity_search_with_relevance_scores`.

## Step 3 · `build_retriever()`: two lanes, fused

- **Dense lane**, `store.as_retriever(search_kwargs={"k": 4})`: the chunks whose vectors sit closest to the question's vector. It catches paraphrases ("send back" matches "return").
- **Keyword lane**, `BM25Retriever`: exact terms. It catches order ids, error codes, product names and rare words, which embeddings tend to blur together.
- **Fusion**, `EnsembleRetriever(weights=[0.5, 0.5])`: **weighted reciprocal rank fusion**, $\text{score}(d) = \sum_i \frac{w_i}{c + \text{rank}_i(d)}$ with $c = 60$. It fuses **ranks, not scores**, because a BM25 score of 7.3 and an L2 distance of 0.9 are not on the same scale (`rg45`).

One improvement over the card is worth making. `BM25Retriever`'s default tokenizer is plain `str.split()`, which is case-sensitive and keeps punctuation attached. Pass a real tokenizer:

In [ ]:
def tokenize(text):                               # lower-case words; keeps codes like e-5003 in one piece
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

def build_retriever(chunks, store, k=TOP_K):
    dense = store.as_retriever(search_kwargs={"k": k})
    sparse = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
    sparse.k = k
    return EnsembleRetriever(retrievers=[dense, sparse], weights=[0.5, 0.5])

retriever = build_retriever(chunks, store)
print("RRF constant c =", retriever.c)

q = "what does error e-5003 mean?"
default_bm25 = BM25Retriever.from_documents(chunks)                       # str.split tokenizer
ours_bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
print("str.split tokens:", default_bm25.preprocess_func(q))
print("our tokens      :", tokenize(q))
default_top = default_bm25.invoke(q)[0].metadata["source"]
ours_top = ours_bm25.invoke(q)[0].metadata["source"]
print(f"keyword lane #1, default tokenizer: {default_top}")
print(f"keyword lane #1, our tokenizer    : {ours_top}")
assert default_top != "error_codes.md" and ours_top == "error_codes.md"

top = retriever.invoke(q)
print(f"\nensemble returned {len(top)} chunks (union of two lanes x k={TOP_K}):",
      [d.metadata["source"] for d in top])
assert top[0].metadata["source"] == "error_codes.md"

Note three things from that output:

- With `str.split`, `error` and `e-5003` do not match `Error` and `E-5003` in the document. The only word that matches anything is `does`, which appears in `warranty.md` ("the warranty does not cover"). The default keyword lane is therefore **confidently wrong**: it puts the warranty page first. With the regex tokenizer, the code matches and `error_codes.md` wins.
- **`BM25Retriever` never returns an empty list.** It always hands back `k` documents, even when every score is zero. Empty retrieval therefore has to be caught elsewhere (Step 4).
- **The ensemble returns the union of both lanes**, up to `2 × k` chunks, not `k`. `answer()` below cuts that list to `TOP_K`. Five tightly relevant chunks beat ten loosely relevant ones (context precision, `rg30`).

## Step 4 · The grounded prompt, `format_docs`, `answer()`

The prompt sets three rules. Answer from the context only. Cite the filename in brackets after each claim. Say you don't know rather than fill the gap from memory. `format_docs` puts `[filename]` above each chunk, which is how the model knows what to cite.

In [ ]:
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using the context below and nothing else. After each claim, cite the "
     "source filename in square brackets. If the context does not answer the "
     "question, say you do not know - do not fill the gap from memory."),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

def answer(question, retriever):
    docs = retriever.invoke(question)[:TOP_K]
    if not docs:                                  # empty retrieval is a real case, not an edge case
        return "Nothing in the knowledge base covers that.", []
    chain = PROMPT | llm | StrOutputParser()
    text = chain.invoke({"question": question, "context": format_docs(docs)})
    return text, docs

print(format_docs(retriever.invoke("How much does express shipping cost?")[:2])[:420], "...")

## Step 5 · Ask it things

Each answer is printed with its sources. The last line of the loop is a production habit worth mentioning: **check that every cited filename was actually retrieved**, because a model can invent a citation (`rg24`).

With `OPENAI_API_KEY` set you get real `gpt-4.1-mini` answers: fluent sentences with `[file.md]` tags. Offline, the stand-in answers *extractively*. It copies the context sentence that best matches the question and keeps its `[source]` tag, so the retrieval really is being tested either way.

In [ ]:
def cited_files(text):
    inside = re.findall(r"\[([^\]]+)\]", text)
    return {name.strip() for group in inside for name in re.split(r"[,;]", group) if name.strip().endswith(".md")}

QUESTIONS = {
    "How many days do I have to ask for a refund?": "refunds.md",
    "What does error E-5003 mean?": "error_codes.md",
    "How much does express shipping cost?": "shipping.md",
    "Who is the CEO of the company?": None,           # not in the knowledge base
}
for question, expected in QUESTIONS.items():
    text, used = answer(question, retriever)
    sources = sorted({d.metadata["source"] for d in used})
    print(f"Q: {question}\nA: {text}\n   sources: {sources}\n")
    if expected:
        assert used[0].metadata["source"] == expected, (question, sources)
    assert cited_files(text) <= set(sources), "the answer cites a file that was not retrieved"

With the CEO question, retrieval still returned four chunks, because retrieval always returns *something*. The **prompt's "say you do not know" rule** is what stopped a made-up answer. This is the low-relevance case of `rg25`. The **truly empty** case happens once filters are in play, for example a user with no permitted documents or a filter that matches nothing. `answer()` catches it before any LLM call, which is both free and hallucination-proof:

In [ ]:
nothing = store.as_retriever(search_kwargs={"k": TOP_K, "filter": {"source": "no_such_file.md"}})
text, used = answer("What is the refund window?", nothing)
print(text, used)
assert used == [] and text.startswith("Nothing in the knowledge base")

## The same skeleton without a framework (~40 lines)

A good follow-up to expect: *"what is LangChain actually doing for you?"* (`pd10`). The strongest answer is to write it without LangChain. The pieces are the OpenAI SDK, NumPy, `rank_bm25` and RRF:

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi

SYSTEM = PROMPT.messages[0].prompt.template        # the same rules as above

class TinyRAG:
    def __init__(self, kb_dir, k=TOP_K, size=800, overlap=120):
        self.k, self.chunks = k, []
        for path in sorted(Path(kb_dir).rglob("*.md")):            # load
            text = path.read_text(encoding="utf-8")
            for start in range(0, max(len(text) - overlap, 1), size - overlap):   # fixed windows
                self.chunks.append({"source": path.name, "text": text[start:start + size]})
        texts = [c["text"] for c in self.chunks]
        self.vecs = self.embed(texts)                              # embed: (n, d), unit length
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])        # keyword index

    def embed(self, texts):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in resp.data], dtype=np.float32)
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    def retrieve(self, question, c=60):
        dense = np.argsort(-(self.vecs @ self.embed([question])[0]))[: self.k]   # cosine similarity
        sparse = np.argsort(-self.bm25.get_scores(tokenize(question)))[: self.k]
        fused = {}
        for ranking in (dense, sparse):                            # reciprocal rank fusion
            for rank, i in enumerate(ranking, start=1):
                fused[int(i)] = fused.get(int(i), 0.0) + 1.0 / (c + rank)
        return [self.chunks[i] for i in sorted(fused, key=fused.get, reverse=True)[: self.k]]

    def answer(self, question):
        hits = self.retrieve(question)
        context = "\n\n".join(f"[{h['source']}]\n{h['text']}" for h in hits)
        resp = client.chat.completions.create(model=MODEL, temperature=0, messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"}])
        return resp.choices[0].message.content, sorted({h["source"] for h in hits})

In [ ]:
tiny = TinyRAG(KB_DIR)
print(f"TinyRAG: {len(tiny.chunks)} fixed-window chunks (LangChain made {len(chunks)} boundary-aware ones)\n")
for question, expected in QUESTIONS.items():
    text, sources = tiny.answer(question)
    print(f"Q: {question}\nA: {text}\n")
    if expected:
        assert tiny.retrieve(question)[0]["source"] == expected

Same answers, no framework. This is the mapping to say out loud:

| LangChain piece | What it really is |
|---|---|
| `DirectoryLoader` + `TextLoader` | `Path.rglob` + `read_text`, with many file types and error handling |
| `RecursiveCharacterTextSplitter` | a sliding window that prefers paragraph, then line, then word boundaries. TinyRAG's fixed windows cut mid-word |
| `OpenAIEmbeddings` | `client.embeddings.create`, batched, with retries and length checks |
| `FAISS` | a matrix of vectors plus `argsort` of dot products, made fast for millions of rows, saved to disk |
| `BM25Retriever` | `BM25Okapi.get_scores` |
| `EnsembleRetriever` | the RRF loop: `sum(w / (60 + rank))` |
| `PROMPT \| llm \| StrOutputParser()` | an f-string, `client.chat.completions.create`, `.choices[0].message.content` |

**Why use the framework, then?** Uniform interfaces make swaps one-liners (Chroma for FAISS, a local embedder for OpenAI). You also get loaders for 100 file types, plus streaming, batching, async and tracing through the same `.invoke`. **Why skip it?** Fewer dependencies, no import-path churn like the one above, and every line is yours to debug. In the interview, say that you could do either and why you chose one.

## The follow-ups they will ask: "what would you add second?"

The honest order, from the card:

1. **An eval set** of 30–50 real questions with known answers, so every later change can be measured.
2. **A cross-encoder reranker** over a wider candidate pool. On real corpora this is usually the biggest single quality jump.
3. **Metadata and permission filters applied *before* retrieval**, so a user never sees a chunk they may not read.
4. **Incremental re-indexing keyed on a content hash**, because rebuilding everything on each edit stops working at around 10k chunks.

Two smaller ones to volunteer: BM25 is rebuilt in memory at startup, which does not survive a large corpus, and 800/120 are guesses until the eval set weighs in.

### 1 · A small eval set: measure before you change anything

Retrieval hit rate is the first number to track: *is the right file in the top k?* The questions below mix keyword-style ones (codes, plan names) with paraphrases ("send back" instead of "refund", "rejected" instead of "declined").

In [ ]:
EVAL_SET = [   # (question, file that answers it)
    ("How many days do I have to ask for a refund?", "refunds.md"),
    ("What does error E-5003 mean?", "error_codes.md"),
    ("How much does express shipping cost?", "shipping.md"),
    ("Can I use two promo codes on one order?", "faq.md"),
    ("How long is a gift card valid?", "faq.md"),
    ("How long do password reset links last?", "account_security.md"),
    ("Does the warranty cover water damage?", "warranty.md"),
    ("How much is the Pro plan?", "pricing_plans.md"),
    ("Can I send back something I bought on clearance?", "refunds.md"),
    ("My card was rejected at checkout, what is going on?", "error_codes.md"),
    ("I forgot my login, how long is the reset email good for?", "account_security.md"),
    ("My laptop screen broke after I dropped it, is that covered?", "warranty.md"),
    ("Do I get a discount if I pay annually?", "pricing_plans.md"),
]

def gold_ranks(retrieve):
    """retrieve(question) -> ranked filenames. Returns the rank of the right file per question (None = missed)."""
    out = []
    for question, gold in EVAL_SET:
        ranked = list(dict.fromkeys(retrieve(question)))          # dedupe files, keep order
        out.append(ranked.index(gold) + 1 if gold in ranked else None)
    return out

def summarize(ranks, k=3):
    n = len(ranks)
    return {"hit@1": sum(r == 1 for r in ranks) / n,
            f"hit@{k}": sum(r is not None and r <= k for r in ranks) / n,
            "MRR": sum(1 / r for r in ranks if r) / n}

sources = lambda docs: [d.metadata["source"] for d in docs]
dense_only = store.as_retriever(search_kwargs={"k": 10})
bm25_only = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=10)
pool = EnsembleRetriever(retrievers=[dense_only, bm25_only], weights=[0.5, 0.5])   # the wide pool
RANKS = {
    "dense": gold_ranks(lambda q: sources(dense_only.invoke(q))),
    "BM25": gold_ranks(lambda q: sources(bm25_only.invoke(q))),
    "hybrid": gold_ranks(lambda q: sources(pool.invoke(q))),
}
for name, ranks in RANKS.items():
    print(f"{name:8}", "  ".join(f"{m}={v:.2f}" for m, v in summarize(ranks).items()))

### 2 · A reranker hook: retrieve wide, rerank, keep a few

A **cross-encoder** reads the question and a chunk *together* and outputs one relevance score. It is far more accurate than comparing two separately computed vectors, and far too slow to run over the whole corpus. The standard shape is: the first stage fetches a **pool** (20–50 chunks), the cross-encoder scores each `(question, chunk)` pair, and you keep the top 4.

The hook below is a plain function, `rerank(question, docs) -> docs`, so `answer()` does not care which reranker sits behind it. It uses `cross-encoder/ms-marco-MiniLM-L6-v2` (90 MB, runs on CPU, cached locally), and falls back to word overlap if the model is not available.

In [ ]:
def load_hf(cls, name):
    """Use the cached copy without network round-trips; download only if it is not cached."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        return cls(name)

try:
    from sentence_transformers import CrossEncoder
    cross_encoder = load_hf(CrossEncoder, "cross-encoder/ms-marco-MiniLM-L6-v2")
    def pair_scores(question, docs):
        return list(cross_encoder.predict([(question, d.page_content) for d in docs]))
    USING_CROSS_ENCODER = True
    print("reranker: cross-encoder/ms-marco-MiniLM-L6-v2")
except Exception as err:                                   # no model on this machine: degrade, don't crash
    USING_CROSS_ENCODER = False
    print("cross-encoder unavailable, falling back to word overlap:", type(err).__name__)
    def pair_scores(question, docs):
        q = set(tokenize(question))
        return [len(q & set(tokenize(d.page_content))) for d in docs]

def rerank(question, docs, top_n=TOP_K):
    scores = pair_scores(question, docs)
    ranked = sorted(zip(scores, range(len(docs)), docs), key=lambda t: (-t[0], t[1]))
    return [d for _, _, d in ranked[:top_n]]

class RerankedRetriever:
    """Anything with .invoke(question) works in answer(): wide pool -> rerank -> top_n."""
    def __init__(self, first_stage, top_n=TOP_K):
        self.first_stage, self.top_n = first_stage, top_n
    def invoke(self, question):
        return rerank(question, self.first_stage.invoke(question), self.top_n)

reranked = RerankedRetriever(pool)
q = "Can I send back something I bought on clearance?"
print("\nfirst stage :", sources(pool.invoke(q))[:3])
print("reranked    :", sources(reranked.invoke(q))[:3])
RANKS["+rerank"] = gold_ranks(lambda q: sources(reranked.invoke(q)))
assert sources(reranked.invoke(q))[0] == "refunds.md" or not USING_CROSS_ENCODER

print(f"\n{'rank of the right file (- = missed)':56}" + "".join(f"{n:>8}" for n in RANKS))
for i, (question, gold) in enumerate(EVAL_SET):
    print(f"{question[:55]:56}" + "".join(f"{RANKS[n][i] or '-':>8}" for n in RANKS))
for name, ranks in RANKS.items():
    print(f"{name:8}", "  ".join(f"{m}={v:.2f}" for m, v in summarize(ranks).items()))

In [ ]:
names = list(RANKS)
scores = {n: summarize(RANKS[n]) for n in names}
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 3.6))
for j, metric in enumerate(["hit@1", "hit@3", "MRR"]):
    ax.bar(x + (j - 1) * 0.26, [scores[n][metric] for n in names], width=0.26, label=metric)
ax.set_xticks(x, names); ax.set_ylim(0, 1.05); ax.set_ylabel("score")
ax.set_title(f"Retrieval quality on a {len(EVAL_SET)}-question eval set (higher is better)")
ax.legend(ncol=3, loc="lower right"); plt.tight_layout(); plt.show()

Read the chart as an engineer, not a salesperson:

- **The lanes fail on different questions** (see the per-question table), and fusing them is the point of hybrid search (`rg07`). Offline, the stand-in's "dense" vectors are hashed words with a small synonym table, so they behave like a slightly smarter BM25 here. With real OpenAI embeddings the dense lane does noticeably better on the paraphrases.
- **The reranker fixes the clearance question** (`refunds.md` moves from rank 3 to rank 1) and lifts the annual-discount one. It does not win everywhere, though: the dropped-laptop question slips from rank 1 to rank 2. On a 10-chunk toy corpus the first stage is already nearly right, so there is little left to fix. On a real corpus with thousands of near-miss chunks, the reranker is where most of the gain comes from.
- **A reranker cannot fix bad recall** (`rg08`). It only reorders the pool. If the right chunk is not among the 20–50 candidates, no reranker will bring it back, which is why you widen the pool first.
- This is exactly **why the eval set comes first**. It tells you whether the reranker is worth its ~50–100 ms per query on *your* data.

### 3 · Permission filters *before* retrieval

Remember `internal/refund_exceptions.md`? The skeleton indexed it along with everything else, so a customer can retrieve it:

In [ ]:
customer_q = "Can I get a refund after the 30-day window?"
text, used = answer(customer_q, retriever)
print("NO FILTER ->", text)
print("   sources:", sorted({d.metadata["source"] for d in used}))

def build_chunks_with_acl(kb_dir=KB_DIR):
    """build_chunks plus an 'audience' label, taken from the folder the file sits in."""
    docs = DirectoryLoader(str(kb_dir), glob="**/*.md", loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
    for d in docs:                                # metadata set here is copied onto every chunk
        rel = Path(d.metadata["source"]).relative_to(kb_dir)
        d.metadata["audience"] = "internal" if rel.parts[0] == "internal" else "public"
        d.metadata["source"] = rel.name
    return RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120,
                                          add_start_index=True).split_documents(docs)

acl_chunks = build_chunks_with_acl()
acl_store = FAISS.from_documents(acl_chunks, embeddings)

def retriever_for(groups, k=TOP_K):
    """Filter INSIDE both lanes, so forbidden chunks never become candidates."""
    allowed = [c for c in acl_chunks if c.metadata["audience"] in groups]
    dense = acl_store.as_retriever(search_kwargs={"k": k, "fetch_k": 50,
                                                  "filter": {"audience": {"$in": list(groups)}}})
    sparse = BM25Retriever.from_documents(allowed, preprocess_func=tokenize, k=k)
    return EnsembleRetriever(retrievers=[dense, sparse], weights=[0.5, 0.5])

text, used = answer(customer_q, retriever_for({"public"}))
print("\nPUBLIC USER ->", text)
print("   sources:", sorted({d.metadata["source"] for d in used}))
assert all(d.metadata["audience"] == "public" for d in used)
staff_sources = {d.metadata["source"] for d in retriever_for({"public", "internal"}).invoke(customer_q)}
assert "refund_exceptions.md" in staff_sources

The rule: **the filter runs inside retrieval, never after generation.** Once a forbidden chunk has been in the prompt, the model may paraphrase it, and no output filter can reliably catch a paraphrase (`rg29`).

- FAISS in LangChain filters *after* an over-fetch (`fetch_k` candidates, then drop the forbidden ones). With many forbidden chunks you can end up with fewer than `k` results. Qdrant, pgvector (`WHERE` clause), OpenSearch and Pinecone filter **inside** the index search, which is what you want at scale.
- The ACL belongs in the **chunk metadata at index time**, copied from the source system (for example the SharePoint or Confluence permissions). Look up the user's groups at query time from your auth layer, never from the prompt.

### 4 · Incremental re-indexing by content hash

Rebuilding the whole index on every edit re-embeds (and re-pays for) every chunk. Instead, give each chunk a **stable id = hash of its content**. On each sync, diff the ids you want against the ids the store has: add the new ones, delete the stale ones, and leave the rest alone.

In [ ]:
import hashlib

def chunk_id(c):
    return hashlib.sha256(f"{c.metadata['source']}|{c.page_content}".encode()).hexdigest()[:16]

def sync_index(store, chunks):
    wanted = {chunk_id(c): c for c in chunks}
    have = set(store.index_to_docstore_id.values())
    stale = sorted(have - wanted.keys())
    fresh = [cid for cid in wanted if cid not in have]
    if stale:
        store.delete(stale)
    if fresh:
        store.add_documents([wanted[cid] for cid in fresh], ids=fresh)
    return {"added": len(fresh), "deleted": len(stale), "unchanged": len(wanted) - len(fresh)}

inc_store = FAISS.from_documents(chunks, embeddings, ids=[chunk_id(c) for c in chunks])
n0 = embeddings.texts_embedded
print("nothing changed :", sync_index(inc_store, build_chunks()), "| texts embedded:", embeddings.texts_embedded - n0)

(KB_DIR / "refunds.md").write_text(KB_FILES["refunds.md"].replace("30 days", "45 days") + "\n", encoding="utf-8")
n0 = embeddings.texts_embedded
new_chunks = build_chunks()
report = sync_index(inc_store, new_chunks)
print("refunds.md edited:", report, "| texts embedded:", embeddings.texts_embedded - n0)
assert report == {"added": 1, "deleted": 1, "unchanged": len(new_chunks) - 1}
assert embeddings.texts_embedded - n0 == 1                  # only the changed chunk was re-embedded

text, _ = answer("How many days do I have to ask for a refund?", build_retriever(new_chunks, inc_store))
print("\nafter the sync  :", text)

Only the edited chunk was re-embedded. Three production notes to add:

- **BM25 has to be rebuilt too.** Here it lives in memory and rebuilding costs milliseconds. At scale, the keyword index lives in OpenSearch or Elasticsearch and gets the same upsert and delete treatment.
- **Deletions come from the source of truth.** Run the sync from a change feed (webhooks, CDC, a nightly listing), and treat "file gone" as delete-all-its-chunks (`rg40`).
- LangChain ships this pattern as `langchain_core.indexing.index(docs, record_manager, vector_store, cleanup="incremental", source_id_key="source")`, with a `RecordManager` (SQL-backed in production) that remembers the hashes. It is worth naming, but in an interview the 12 lines above show you understand what it does.

## The complete `rag.py`

This is the whole file, ready to paste into an editor. It uses the same functions as the steps above, with the two model lines written out for OpenAI. The `tokenize` tweak and the `[:TOP_K]` slice are the two small improvements over the card. Run it with `python rag.py` next to a `knowledge_base/` folder.

```python
"""rag.py - a knowledge-base folder in, grounded answers with sources out.

pip install langchain langchain-community langchain-text-splitters langchain-openai \
            langchain-classic faiss-cpu rank_bm25
env: OPENAI_API_KEY
"""
import re
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever    # LangChain 1.x: not langchain.retrievers
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

KB_DIR = Path("knowledge_base")
DB_DIR = Path("faiss_index")
TOP_K = 4

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


# ---- 1. INDEX: runs once, and again whenever a document changes ----
def build_chunks(kb_dir=KB_DIR):
    docs = DirectoryLoader(str(kb_dir), glob="**/*.md", loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=120, add_start_index=True
    ).split_documents(docs)
    for c in chunks:                              # the filename is what the answer will cite
        c.metadata["source"] = Path(c.metadata["source"]).name
    return chunks


def build_store(chunks, db_dir=DB_DIR):
    if db_dir.exists():                           # already embedded: do not pay for it twice
        return FAISS.load_local(str(db_dir), embeddings,
                                allow_dangerous_deserialization=True)  # our own file, so safe
    store = FAISS.from_documents(chunks, embeddings)
    store.save_local(str(db_dir))
    return store


# ---- 2. RETRIEVE: meaning lane and keyword lane, fused ----
def tokenize(text):                               # lower-case words; keeps codes like e-5003 in one piece
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def build_retriever(chunks, store, k=TOP_K):
    dense = store.as_retriever(search_kwargs={"k": k})
    sparse = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
    sparse.k = k
    return EnsembleRetriever(retrievers=[dense, sparse], weights=[0.5, 0.5])


# ---- 3. GENERATE: answer from the context, or admit it is not there ----
PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using the context below and nothing else. After each claim, cite the "
     "source filename in square brackets. If the context does not answer the "
     "question, say you do not know - do not fill the gap from memory."),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)


def answer(question, retriever):
    docs = retriever.invoke(question)[:TOP_K]
    if not docs:                                  # empty retrieval is a real case, not an edge case
        return "Nothing in the knowledge base covers that.", []
    chain = PROMPT | llm | StrOutputParser()
    text = chain.invoke({"question": question, "context": format_docs(docs)})
    return text, docs


# ---- 4. ASK ----
if __name__ == "__main__":
    chunks = build_chunks()
    retriever = build_retriever(chunks, build_store(chunks))
    while True:
        q = input("ask> ").strip()
        if not q:
            break
        text, used = answer(q, retriever)
        print(text)
        print("sources:", sorted({d.metadata["source"] for d in used}))
```

## How to explain each line

| Line | What to say |
|---|---|
| `DirectoryLoader(..., glob="**/*.md", loader_cls=TextLoader)` | "Reads every markdown file under the folder into `Document`s: text plus `metadata['source']`." |
| `loader_kwargs={"encoding": "utf-8"}` | "Without it, Windows decodes with cp1252 and non-ASCII characters silently turn into mojibake." |
| `RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)` | "≤ 800 characters per chunk, cut at paragraph, then line, then word boundaries. 120 characters of overlap keep a sentence on the cut whole in one chunk. The eval set decides the final numbers." |
| `add_start_index=True` | "Character offset of the chunk, for highlighting and dedup." |
| `c.metadata["source"] = Path(...).name` | "The filename is what gets cited. A full path wastes tokens and leaks the directory layout." |
| `FAISS.from_documents(chunks, embeddings)` | "Embeds every chunk in batches and builds an exact nearest-neighbour index in memory." |
| `save_local` / `load_local(..., allow_dangerous_deserialization=True)` | "Two files: vectors plus a pickled docstore. Unpickling can run code, so it is opt-in, and only for files I wrote." |
| `store.as_retriever(search_kwargs={"k": k})` | "The meaning lane: top-k by vector distance, where lower distance is closer." |
| `BM25Retriever.from_documents(chunks, preprocess_func=tokenize)` | "The keyword lane, for codes, ids and names. The default tokenizer is `str.split`, so I pass a real one." |
| `EnsembleRetriever(retrievers=[dense, sparse], weights=[0.5, 0.5])` | "Weighted reciprocal rank fusion, `Σ w / (60 + rank)`. It fuses ranks because the two score scales aren't comparable. It moved to `langchain_classic` in 1.x." |
| `ChatPromptTemplate.from_messages([...])` | "System: the rules (context only, cite, admit ignorance). Human: the question, then the context." |
| `format_docs` | "Puts a `[filename]` header on each chunk. That header is what the model cites." |
| `PROMPT \| llm \| StrOutputParser()` | "LCEL: a dict goes in, messages go to the model, an `AIMessage` comes back and ends up as a `str`. Every piece is a Runnable, so `.stream()`, `.batch()` and `.ainvoke()` work for free." |
| `retriever.invoke(question)[:TOP_K]` | "The ensemble returns up to 2k chunks. I keep the best k, because precision in the context matters." |
| `if not docs: return ...` | "No context means no LLM call. It is cheaper, and it avoids a hallucination." |
| `temperature=0` | "Repeatable answers, which makes eval and debugging much easier." |

## Try it yourself

**1.** Change the chunking to `chunk_size=300, chunk_overlap=50`. How many chunks do you get now, and does `faq.md` still produce a heading-only chunk?

In [ ]:
# Solution — try it yourself first, then run this
small = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, add_start_index=True).split_documents(
    DirectoryLoader(str(KB_DIR), glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}).load())
print(f"800/120 -> {len(chunks)} chunks;  300/50 -> {len(small)} chunks")
tiny_chunks = [c.page_content for c in small if len(c.page_content) < 40]
print("chunks under 40 characters:", tiny_chunks)

**2.** Swap the embedder for a **local** model without touching any other function: implement the two-method `Embeddings` interface around `sentence-transformers/all-MiniLM-L6-v2`, rebuild the store, and ask the clearance question. What else *must* change when you swap embedding models? (Answer: the index has to be rebuilt. Old vectors come from a different model and a different space, 1536-d versus 384-d.)

In [ ]:
# Solution — try it yourself first, then run this
try:
    from sentence_transformers import SentenceTransformer

    class LocalEmbeddings(Embeddings):
        def __init__(self, name="sentence-transformers/all-MiniLM-L6-v2"):
            self.model = load_hf(SentenceTransformer, name)
        def embed_documents(self, texts):
            return self.model.encode(texts, normalize_embeddings=True).tolist()
        def embed_query(self, text):
            return self.embed_documents([text])[0]

    local_store = FAISS.from_documents(new_chunks, LocalEmbeddings())      # new vectors: a rebuild, not a load
    print("vector dim:", local_store.index.d, "(the OpenAI-style index was", store.index.d, ")")
    q = "Can I send back something I bought on clearance?"
    print("dense lane, MiniLM:", sources(local_store.as_retriever(search_kwargs={"k": 3}).invoke(q)))
    print("hybrid,     MiniLM:", sources(build_retriever(new_chunks, local_store).invoke(q))[:3])
except Exception as err:
    print("sentence-transformers model not available here:", type(err).__name__)

**3.** A user types *"what does e-7001 mean"* (lower case, no question mark). Rank the chunks with BM25 three ways: (a) the default `str.split` tokenizer, (b) `tokenize`, (c) `tokenize` with a few stopwords removed. Which file comes first each time? Why does (b) get it wrong on this tiny corpus?

In [ ]:
# Solution — try it yourself first, then run this
q = "what does e-7001 mean"
STOP = {"what", "does", "do", "is", "the", "a", "an", "i", "my", "mean", "means", "how", "can"}
tokenize_nostop = lambda text: [t for t in tokenize(text) if t not in STOP]

tops = {}
for name, fn in [("(a) str.split", None), ("(b) tokenize", tokenize), ("(c) tokenize - stopwords", tokenize_nostop)]:
    bm = BM25Retriever.from_documents(new_chunks, **({"preprocess_func": fn} if fn else {}))
    tops[name] = bm.invoke(q)[0].metadata["source"]
    print(f"{name:26} top = {tops[name]:18} scores>0: {int((bm.vectorizer.get_scores(bm.preprocess_func(q)) > 0).sum())}")
assert tops["(c) tokenize - stopwords"] == "error_codes.md"
# (a) "e-7001" does not match "E-7001"; the only hit is "does" in warranty.md, so the warranty page wins.
# (b) the code matches now, but "does" occurs in exactly one chunk ("the warranty does not cover"), so on a
#     10-chunk corpus its IDF is as high as the code's, and the shorter warranty chunk wins on length normalisation.
#     On a real corpus "does" is everywhere and its IDF falls to ~0; a stopword list is cheap insurance anyway.
# (c) only "e-7001" is left, and it points straight at error_codes.md.

## Say it in an interview

**The 30-second answer (`rg00`):** "Python and LangChain. At index time I load the folder with `DirectoryLoader`, split it with `RecursiveCharacterTextSplitter` at 800 characters with 120 overlap, keep the filename as `source` metadata, embed with `text-embedding-3-small`, and save a FAISS index. At query time I use hybrid retrieval: FAISS for meaning plus BM25 for exact terms, fused with reciprocal rank fusion via `EnsembleRetriever`. I keep the top 4, and the prompt says *answer only from the context, cite the filename in brackets, say you don't know otherwise*. The function returns the answer and its sources, and short-circuits if retrieval comes back empty."

**Key numbers:** 800/120 characters ≈ 200/30 tokens as a starting point · 4–5 chunks in the prompt · RRF `c = 60` · rerank a pool of 20–50 down to 5 · `text-embedding-3-small` is 1536-d, MiniLM is 384-d · full rebuilds stop being acceptable at around 10k chunks.

**Traps they probe:**
- `from langchain.retrievers import EnsembleRetriever` fails in LangChain 1.x. It lives in `langchain_classic.retrievers`.
- FAISS scores are **distances** (lower is better). Treating them as similarities flips your threshold logic.
- `BM25Retriever` tokenizes with `str.split` by default and **never returns an empty list**.
- `EnsembleRetriever` returns up to `2k` documents, not `k`.
- `load_local` needs `allow_dangerous_deserialization=True` because the docstore is a pickle. Only load files you wrote, and only with the same embedding model you built with.
- Filtering permissions *after* generation is a data leak. Filter inside retrieval.
- The `if Path(DB_DIR).exists()` cache loads a **stale** index after an edit. Fix it with content-hash sync.

**"What would you add second?"** An eval set, then a reranker over a wider pool, then permission filters before retrieval, then incremental re-indexing by content hash. That order puts measurement first, then the biggest quality gain, then safety, then scale.

## Next

- Next round: [Build an agent in 30 minutes](02_build_an_agent_in_30_minutes.ipynb) · [Semantic search service](04_semantic_search_service.ipynb) · [Eval harness from scratch](05_eval_harness_from_scratch.ipynb)
- Each piece in depth: [chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) · [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb) · [FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) · [hybrid search and RRF](../09_rag/04_hybrid_search_and_rrf.ipynb) · [reranking](../09_rag/05_reranking.ipynb) · [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb)
- The LangChain side: [loaders, splitters, vector stores, retrievers](../13_langchain/04_loaders_splitters_vectorstores_retrievers.ipynb) · [RAG chain end to end](../13_langchain/05_rag_chain_end_to_end.ipynb)
- Measuring it: [RAG evaluation: retrieval and faithfulness](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)
- Theory: [interview bank, card `rg00`](../../ai_interview_prep/index.html) · [genai_flow course](../../genai_flow/index.html) · [langchain course](../../langchain/index.html)